# 한국투자증권 모의투자 API 확인

이 노트북은 `api/.env` 설정을 읽어서 토큰, 국내주식 잔고, 국내주식 현재가를 확인합니다.

키와 토큰은 화면에 전체 출력하지 않도록 마스킹합니다.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "api":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from api.kis_client import KisClient

client = KisClient.from_env(str(PROJECT_ROOT / "api" / ".env"))
config = client.config

print("env:", config.env)
print("base_url:", config.base_url)
print("domestic_stock:", config.domestic_stock)
print("domestic_futures:", config.domestic_futures)

env: paper
base_url: https://openapivts.koreainvestment.com:29443
domestic_stock: KisAccount(account_no='50189904', product_code='01')
domestic_futures: KisAccount(account_no='60043529', product_code='03')


## 1. 접근 토큰 확인

한국투자증권은 접근토큰 발급이 1분당 1회로 제한될 수 있습니다. 클라이언트가 `api/.token_cache.json`을 사용해서 가능하면 기존 토큰을 재사용합니다.

In [2]:
def mask(value: str, left: int = 6, right: int = 6) -> str:
    if not value or len(value) <= left + right:
        return "***"
    return f"{value[:left]}...{value[-right:]}"

token = client.access_token
print("access_token:", mask(token))

access_token: eyJ0eX...DzmfVw


## 2. 국내주식 모의계좌 잔고

In [3]:
balance = client.get_domestic_stock_balance()
balance

{'ctx_area_fk100': '                                                                                                    ',
 'ctx_area_nk100': '                                                                                                    ',
 'output1': [],
 'output2': [{'dnca_tot_amt': '500000000',
   'nxdy_excc_amt': '500000000',
   'prvs_rcdl_excc_amt': '500000000',
   'cma_evlu_amt': '0',
   'bfdy_buy_amt': '0',
   'thdt_buy_amt': '0',
   'nxdy_auto_rdpt_amt': '0',
   'bfdy_sll_amt': '0',
   'thdt_sll_amt': '0',
   'd2_auto_rdpt_amt': '0',
   'bfdy_tlex_amt': '0',
   'thdt_tlex_amt': '0',
   'tot_loan_amt': '0',
   'scts_evlu_amt': '0',
   'tot_evlu_amt': '500000000',
   'nass_amt': '500000000',
   'fncg_gld_auto_rdpt_yn': '',
   'pchs_amt_smtl_amt': '0',
   'evlu_amt_smtl_amt': '0',
   'evlu_pfls_smtl_amt': '0',
   'tot_stln_slng_chgs': '0',
   'bfdy_tot_asst_evlu_amt': '500000000',
   'asst_icdc_amt': '0',
   'asst_icdc_erng_rt': '0.00000000'}],
 'rt_cd': '0',
 'msg_cd': '203

In [4]:
summary = balance.get("output2", [{}])[0]

print("예수금:", summary.get("dnca_tot_amt"))
print("총평가금액:", summary.get("tot_evlu_amt"))
print("순자산:", summary.get("nass_amt"))
print("평가손익:", summary.get("evlu_pfls_smtl_amt"))
print("응답메시지:", balance.get("msg1"))

예수금: 500000000
총평가금액: 500000000
순자산: 500000000
평가손익: 0
응답메시지: 모의투자 조회가 완료되었습니다.                                                 


## 3. 국내주식 현재가

`stock_code`를 바꾸면 다른 종목도 조회할 수 있습니다.

In [5]:
stock_code = "005930"
price = client.get_domestic_stock_price(stock_code)
price

{'output': {'iscd_stat_cls_code': '55',
  'marg_rate': '60.00',
  'rprs_mrkt_kor_name': 'KOSPI200',
  'new_hgpr_lwpr_cls_code': '신고가',
  'bstp_kor_isnm': '전기·전자',
  'temp_stop_yn': 'N',
  'oprc_rang_cont_yn': 'N',
  'clpr_rang_cont_yn': 'N',
  'crdt_able_yn': 'Y',
  'grmn_rate_cls_code': '60',
  'elw_pblc_yn': 'Y',
  'stck_prpr': '299000',
  'prdy_vrss': '6500',
  'prdy_vrss_sign': '2',
  'prdy_ctrt': '2.22',
  'acml_tr_pbmn': '6533787707250',
  'acml_vol': '21773154',
  'prdy_vrss_vol_rate': '118.36',
  'stck_oprc': '298000',
  'stck_hgpr': '302000',
  'stck_lwpr': '297500',
  'stck_mxpr': '380000',
  'stck_llam': '205000',
  'stck_sdpr': '292500',
  'wghn_avrg_stck_prc': '300084.74',
  'hts_frgn_ehrt': '48.32',
  'frgn_ntby_qty': '-53250',
  'pgtr_ntby_qty': '1615701',
  'pvt_scnd_dmrs_prc': '303500',
  'pvt_frst_dmrs_prc': '298000',
  'pvt_pont_val': '295000',
  'pvt_frst_dmsp_prc': '289500',
  'pvt_scnd_dmsp_prc': '286500',
  'dmrs_val': '296500',
  'dmsp_val': '288000',
  'cpfn': 

In [6]:
output = price.get("output", {})

print("종목코드:", output.get("stck_shrn_iscd"))
print("현재가:", output.get("stck_prpr"))
print("전일대비:", output.get("prdy_vrss"))
print("등락률:", output.get("prdy_ctrt"))
print("거래량:", output.get("acml_vol"))
print("응답메시지:", price.get("msg1"))

종목코드: 005930
현재가: 299000
전일대비: 6500
등락률: 2.22
거래량: 21773154
응답메시지: 정상처리 되었습니다.


## 4. 삼성전자 1주 시장가 매수

아래 셀은 실제로 모의투자 주문을 넣는 셀입니다. 기본값은 `RUN_MARKET_BUY = False`라서 실행해도 주문이 나가지 않습니다.

주문을 넣으려면 주문 내용을 확인한 뒤 `RUN_MARKET_BUY = True`로 바꾸고 셀을 실행하세요.

In [ ]:
market_buy_order = {
    "stock_code": "005930",
    "stock_name": "삼성전자",
    "quantity": 1,
    "order_type": "market",
    "account_no": config.domestic_stock.account_no,
    "account_product_code": config.domestic_stock.product_code,
    "env": config.env,
}

market_buy_order

In [ ]:
RUN_MARKET_BUY = False

if RUN_MARKET_BUY:
    order_result = client.buy_domestic_stock_market(
        stock_code=market_buy_order["stock_code"],
        quantity=market_buy_order["quantity"],
    )
else:
    order_result = {
        "skipped": True,
        "message": "RUN_MARKET_BUY를 True로 바꾸면 삼성전자 1주 시장가 매수 주문을 전송합니다.",
    }

order_result

## 5. 삼성전자 1주 시장가 매도

아래 셀도 실제로 모의투자 주문을 넣는 셀입니다. 기본값은 `RUN_MARKET_SELL = False`라서 실행해도 주문이 나가지 않습니다.

매도 주문을 넣으려면 보유 수량을 확인한 뒤 `RUN_MARKET_SELL = True`로 바꾸고 셀을 실행하세요.

In [ ]:
market_sell_order = {
    "stock_code": "005930",
    "stock_name": "삼성전자",
    "quantity": 1,
    "order_type": "market",
    "account_no": config.domestic_stock.account_no,
    "account_product_code": config.domestic_stock.product_code,
    "env": config.env,
}

market_sell_order

In [ ]:
RUN_MARKET_SELL = False

if RUN_MARKET_SELL:
    order_result = client.sell_domestic_stock_market(
        stock_code=market_sell_order["stock_code"],
        quantity=market_sell_order["quantity"],
    )
else:
    order_result = {
        "skipped": True,
        "message": "RUN_MARKET_SELL을 True로 바꾸면 삼성전자 1주 시장가 매도 주문을 전송합니다.",
    }

order_result

## 6. 선물옵션 계좌 설정 확인

아직 선물옵션 잔고/주문 API 메서드는 붙이기 전입니다. 우선 `.env`에서 계좌가 정상으로 읽히는지만 확인합니다.

In [7]:
config.domestic_futures

KisAccount(account_no='60043529', product_code='03')